## Load the recorded results

Run this notebook from the repository root. The code only opens and reads the CSV; it does not modify it.

In [1]:
import csv
import statistics
from collections import defaultdict
from pathlib import Path

results_path = Path("TP3/results.csv")
if not results_path.is_file():
    raise FileNotFoundError(
        f"Cannot find {results_path}. Run this notebook from the repository root."
    )

with results_path.open(newline="", encoding="utf-8-sig") as csv_file:
    reader = csv.DictReader(csv_file)
    required_columns = {
        "config_id", "Language", "Repetitions", "DistributionCoverage",
        "RandomNumberGenerator", "TypeOfProblem", "run_number", "seed",
        "answer_percent", "actual_engine", "runner_os", "python_version",
        "architecture",
    }
    missing = required_columns - set(reader.fieldnames or [])
    if missing:
        raise ValueError(f"Results CSV is missing columns: {sorted(missing)}")
    results = list(reader)

if not results:
    raise ValueError(f"No result rows found in {results_path}")

for row in results:
    row["answer_percent"] = float(row["answer_percent"])
    row["run_number"] = int(row["run_number"])
    row["seed"] = int(row["seed"])

print(f"Read {len(results)} result rows from {results_path}")

Read 378 result rows from TP3/results.csv


## Comparison matrix and analysis method

The workflow matrix varies operating system/CPU architecture and Python version. The sample also varies repetition count, input distribution, random-number generator, and problem identity. C++ configurations are skipped by the Python runner, so this CSV describes Python variants only.

For the feature comparisons below, the notebook uses one environment as a reference instead of counting identical copies from every matrix job as independent observations. It first checks whether answers and engines match across environments for each configuration/run. This is a descriptive comparison, not a fully balanced factorial experiment: the randomly sampled factors are not evenly represented, and some combinations do not occur.

In [2]:
environments = sorted({
    (row["runner_os"], row["python_version"], row["architecture"])
    for row in results
})
runs_by_config = defaultdict(list)
for row in results:
    key = (
        row["config_id"], row["run_number"], row["Language"],
        row["Repetitions"], row["DistributionCoverage"],
        row["RandomNumberGenerator"], row["TypeOfProblem"],
    )
    runs_by_config[key].append((row["answer_percent"], row["actual_engine"]))

environment_mismatches = sum(
    len(set(values)) > 1 for values in runs_by_config.values()
)
reference_environment = environments[0]
reference_rows = [
    row for row in results
    if (row["runner_os"], row["python_version"], row["architecture"])
    == reference_environment
]

print(f"Sampled feature configurations: {len({r['config_id'] for r in results})}")
print(f"Environment combinations: {len(environments)}")
print(f"Rows per environment: {len(reference_rows)}")
print(f"Config/run records differing across environments: {environment_mismatches}")
print(f"Reference environment for feature summaries: {reference_environment}")
print("Environment matrix:")
for environment in environments:
    print(f"  {environment[0]}, Python {environment[1]}, {environment[2]}")

Sampled feature configurations: 21
Environment combinations: 6
Rows per environment: 63
Config/run records differing across environments: 0
Reference environment for feature summaries: ('ubuntu-24.04', '3.11', 'x64')
Environment matrix:
  ubuntu-24.04, Python 3.11, x64
  ubuntu-24.04, Python 3.12, x64
  ubuntu-24.04, Python 3.13, x64
  ubuntu-24.04-arm, Python 3.11, arm64
  ubuntu-24.04-arm, Python 3.12, arm64
  ubuntu-24.04-arm, Python 3.13, arm64


## Results by variability factor

The summaries below use the reference environment. The standard deviation shown for a factor includes differences between sampled configurations as well as run-to-run variation; it should not be read as a pure estimate of randomness.

In [3]:
def print_group_summary(rows, factor):
    groups = defaultdict(list)
    for row in rows:
        groups[row[factor]].append(row["answer_percent"])
    print(f"\n{factor}: n | mean % | median % | min % | max %")
    for value, answers in sorted(groups.items()):
        print(
            f"{value}: {len(answers)} | {statistics.mean(answers):.4f} | "
            f"{statistics.median(answers):.4f} | {min(answers):.4f} | "
            f"{max(answers):.4f}"
        )

for factor in (
    "DistributionCoverage", "TypeOfProblem", "Repetitions",
    "RandomNumberGenerator",
):
    print_group_summary(reference_rows, factor)

by_distribution_problem = defaultdict(list)
for row in reference_rows:
    key = (row["DistributionCoverage"], row["TypeOfProblem"])
    by_distribution_problem[key].append(row["answer_percent"])

print("\nDistributionCoverage × TypeOfProblem: n | mean % | min % | max %")
for (distribution, problem), answers in sorted(by_distribution_problem.items()):
    print(
        f"{distribution} / {problem}: {len(answers)} | "
        f"{statistics.mean(answers):.4f} | {min(answers):.4f} | {max(answers):.4f}"
    )

repeats_by_config = defaultdict(list)
for row in reference_rows:
    key = (row["config_id"], row["Repetitions"])
    repeats_by_config[key].append(row["answer_percent"])

spreads_by_repetitions = defaultdict(list)
for (_, repetitions), answers in repeats_by_config.items():
    if len(answers) > 1:
        spreads_by_repetitions[repetitions].append(max(answers) - min(answers))

print("\nWithin-configuration range across runs: configurations | mean range (percentage points) | max range")
for repetitions, ranges in sorted(spreads_by_repetitions.items()):
    print(
        f"{repetitions}: {len(ranges)} | {statistics.mean(ranges):.4f} | "
        f"{max(ranges):.4f}"
    )


DistributionCoverage: n | mean % | median % | min % | max %
IntegerZeroToOneThousandAsFloat: 9 | 100.0000 | 100.0000 | 100.0000 | 100.0000
LogUniformMagnitudeRandomSign: 12 | 82.2364 | 82.2827 | 81.8000 | 82.3795
StandardNormal: 12 | 67.0716 | 68.5200 | 54.8158 | 82.0000
SuiteUniformZeroToOne: 9 | 82.7576 | 82.8105 | 81.9800 | 83.7600
UniformMinusOneToOne: 15 | 80.7986 | 80.0000 | 60.2437 | 100.0000
UniformZeroToOneThousand: 6 | 75.6005 | 74.5372 | 60.0000 | 90.0000

TypeOfProblem: n | mean % | median % | min % | max %
Associativity: 45 | 82.7915 | 82.3054 | 60.0000 | 100.0000
Distributivity: 18 | 76.4710 | 65.1867 | 54.8158 | 100.0000

Repetitions: n | mean % | median % | min % | max %
OneMillion: 15 | 77.1610 | 82.3325 | 60.2437 | 100.0000
fifty: 9 | 81.5556 | 76.0000 | 62.0000 | 100.0000
fiveThousand: 18 | 85.9500 | 82.2400 | 68.0600 | 100.0000
ten: 6 | 80.0000 | 80.0000 | 60.0000 | 100.0000
tenMillion: 15 | 78.9054 | 82.3442 | 54.8158 | 100.0000

RandomNumberGenerator: n | mean % 

## Explaining the answers from Labs #1 and #2



## Recommendations and conclusions

- For a stable Monte Carlo estimate, use at least **100,000 repetitions** and a fixed, recorded seed. For tighter precision, use **1,000,000 or more**; the current feature model already offers one million and ten million. Larger counts cost more runtime and cannot remove differences caused by changing the distribution or problem.
- To reproduce the original “about 82%” result, choose `Associativity`, binary64 arithmetic, exact `==`, and independent uniform inputs in `[0, 1)`, then report the repetition count, RNG, and seed. In this model, `SuiteUniformZeroToOne` selects that distribution.
- Compare feature effects using matched configurations: vary one factor while keeping all other factors fixed, and include enough runs. The current random sample is small and unbalanced, so it is useful for exploration but not for isolating every causal effect.
- Record the environment with each run. The current matrix results happen to match across its six environments, but repeat the comparison if the code, dependencies, compiler, or numeric implementation changes.

**Conclusion:** the observed percentage is determined by the floating-point expression and the sampling distribution, while a finite repetition count adds Monte Carlo variability. A large repetition count stabilizes an estimate of a particular experiment; it does not make different experiments converge to one universal percentage.